In [1]:
from ultralytics import YOLO

import torch
import torchvision.transforms as transforms

from torchvision.models import resnet18
import torchvision.models as models
from PIL import Image

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import shutil
from tqdm import tqdm
import time

In [2]:
YOLO_MODEL = YOLO("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\runs\\detect\\runs\\YOLOv8_baseline\\weights\\best.pt")

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = cnn.fc.in_features
cnn.fc = torch.nn.Sequential(
    torch.nn.Linear(num_features, 128),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.5),
    torch.nn.Linear(128, 2)
)

cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\best_chicken_cnn_augument2.pth"))

cnn.to(DEVICE)

cnn.eval()

C:\Users\klanz\AppData\Local\Temp\ipykernel_14424\102316409.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\M

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [5]:
def classify_crop(crop):

    image = Image.fromarray(cv2.cvtColor(crop,cv2.COLOR_BGR2RGB))

    tensor = transform(image)

    tensor = tensor.unsqueeze(0)

    tensor = tensor.to(DEVICE)

    with torch.no_grad():

        prediction = cnn(tensor)

        prediction = prediction.argmax(1).item()

    return prediction

In [6]:
def door_decision(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    cnn_classes = [p["cnn_prediction"] for p in predictions]

    if 1 in cnn_classes:
        return "CLOSE"

    if 0 in cnn_classes:
        return "OPEN"

    return "CLOSE"

In [7]:
def door_decision_yolo(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    classes = [p["class"] for p in predictions]

    if 1 in classes:
        return "CLOSE"

    if 0 in classes:
        return "OPEN"

    return "CLOSE"

In [8]:
CLASS_NAMES = {
    0: "chicken",
    1: "not_chicken"
}

In [9]:
def run_yolo(image_path, conf=0.25):

    result = YOLO_MODEL.predict(
        source=str(image_path),
        conf=conf,
        verbose=False
    )[0]

    predictions = []

    for box in result.boxes:

        cls = int(box.cls.item())

        confidence = float(box.conf.item())

        x1, y1, x2, y2 = box.xyxy.cpu().numpy()[0]

        predictions.append({

            "class": cls,
            "class_name": CLASS_NAMES[cls],
            "confidence": confidence,
            "bbox": [int(x1), int(y1), int(x2), int(y2)]

        })

    return predictions

In [10]:
def run_pipeline(image_path, conf=0.25):

    detections = run_yolo(image_path, conf)

    image = cv2.imread(str(image_path))

    final_predictions = []

    for det in detections:

        x1, y1, x2, y2 = det["bbox"]

        crop = image[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        cnn_prediction = classify_crop(crop)

        final_predictions.append({

            "yolo_prediction": det["class"],

            "cnn_prediction": cnn_prediction,

            "confidence": det["confidence"],

            "bbox": det["bbox"]

        })

    return final_predictions

In [11]:
def load_ground_truth(label_path):

    if not Path(label_path).exists():
        return []

    gt=[]

    with open(label_path) as f:

        for line in f:

            line=line.strip()

            if line=="":

                continue

            cls=int(line.split()[0])

            gt.append(cls)

    return gt

In [12]:
def ground_truth_decision(gt_classes):

    if 1 in gt_classes:
        return "CLOSE"

    if 0 in gt_classes:
        return "OPEN"

    return "CLOSE"

In [13]:
test_images = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset\\test\\images").glob("*"))

results = []

In [14]:
for image_path in test_images:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")

    gt = load_ground_truth(label_path)

    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()

    yolo_predictions = run_yolo(image_path)

    yolo_time = (time.perf_counter()-start)*1000

    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()

    pipeline_predictions = run_pipeline(image_path)

    pipeline_time = (time.perf_counter()-start)*1000

    pipeline_decision = door_decision(pipeline_predictions)

    results.append({

        "image": image_path.name,

        "ground_truth": gt_decision,

        "yolo": yolo_decision,

        "pipeline": pipeline_decision,

        "yolo_time_ms": yolo_time,

        "pipeline_time_ms": pipeline_time,

        "objects_gt": gt,

        "objects_yolo": [x["class"] for x in yolo_predictions],

        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [15]:
df = pd.DataFrame(results)

df.head(20)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,523.4274,70.5751,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,13.5343,20.8689,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,17.4163,25.8749,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,15.3473,21.5892,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,18.2919,40.6289,"[0, 0, 0]","[0, 0, 0, 0, 0]","[0, 0, 0, 1, 1]"
5,1085.jpeg,OPEN,OPEN,OPEN,14.7512,19.6225,[0],[0],[0]
6,109.jpeg,OPEN,OPEN,OPEN,16.5605,22.4777,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,17.5137,23.7270,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,18.0464,23.1031,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,CLOSE,16.9233,33.1047,"[0, 0, 0]","[0, 0, 0]","[0, 0, 1]"


In [16]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

import numpy as np

In [17]:
decision_map = {

    "OPEN":1,

    "CLOSE":0

}

In [18]:
def evaluate_system(df, prediction_column, time_column):

    gt = df["ground_truth"].map(decision_map)

    pred = df[prediction_column].map(decision_map)

    accuracy = accuracy_score(gt, pred)

    precision = precision_score(
        gt,
        pred,
        zero_division=0
    )

    recall = recall_score(
        gt,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        gt,
        pred,
        zero_division=0
    )

    cm = confusion_matrix(gt, pred)

    tn, fp, fn, tp = cm.ravel()

    avg_time = df[time_column].mean()

    print("="*50)

    print(prediction_column)

    print("="*50)

    print(f"Accuracy : {accuracy:.4f}")

    print(f"Precision: {precision:.4f}")

    print(f"Recall   : {recall:.4f}")

    print(f"F1-score : {f1:.4f}")

    print()

    print(f"TP : {tp}")

    print(f"FP : {fp}")

    print(f"TN : {tn}")

    print(f"FN : {fn}")

    print()

    print(f"Średni czas: {avg_time:.2f} ms")

    return {

        "Accuracy":accuracy,

        "Precision":precision,

        "Recall":recall,

        "F1":f1,

        "TP":tp,

        "FP":fp,

        "TN":tn,

        "FN":fn,

        "Time":avg_time

    }

In [19]:
yolo_results = evaluate_system(

    df,

    "yolo",

    "yolo_time_ms"

)

yolo
Accuracy : 0.9943
Precision: 0.9959
Recall   : 0.9938
F1-score : 0.9949

TP : 484
FP : 2
TN : 389
FN : 3

Średni czas: 20.15 ms


In [20]:
pipeline_results = evaluate_system(

    df,

    "pipeline",

    "pipeline_time_ms"

)

pipeline
Accuracy : 0.9818
Precision: 0.9856
Recall   : 0.9815
F1-score : 0.9835

TP : 478
FP : 7
TN : 384
FN : 9

Średni czas: 30.77 ms


In [21]:
comparison = pd.DataFrame(

    [

        yolo_results,

        pipeline_results

    ],

    index=[

        "YOLO",

        "YOLO + CNN"

    ]

)

comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.994305,0.995885,0.99384,0.994861,484,2,389,3,20.145158
YOLO + CNN,0.981777,0.985567,0.98152,0.983539,478,7,384,9,30.774838


In [22]:
dangerous_yolo = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN")

]

In [23]:
dangerous_pipeline = df[

    (df["ground_truth"]=="CLOSE") &

    (df["pipeline"]=="OPEN")

]

In [24]:
print()

print("Krytyczne błędy")

print("----------------")

print("YOLO:",len(dangerous_yolo))

print("YOLO+CNN:",len(dangerous_pipeline))


Krytyczne błędy
----------------
YOLO: 2
YOLO+CNN: 7


In [25]:
improved = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN") &

    (df["pipeline"]=="CLOSE")

]

improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,18.0284,21.2635,[1],"[0, 0]","[1, 1]"


In [26]:
empty_images = 0
false_detections = 0

for row in results:

    if len(row["objects_gt"]) == 0:

        empty_images += 1

        if len(row["objects_yolo"]) > 0:
            false_detections += 1

print("Puste obrazy:",empty_images)
print("Fałszywe detekcje:",false_detections)

if empty_images>0:

    print(
        "Odsetek:",
        false_detections/empty_images
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [27]:
fp_images=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["pipeline"]=="OPEN":

        fp_images.append(row["image"])

print("False Positive:",len(fp_images))

fp_images

False Positive: 7


['coyote__lila_WSU_Lynx_IMG_0965_jpg.rf.vqQg0CptK1hvCGBkRVla.jpg',
 'Image-84-a5ad3f.jpg',
 'Image-88-8f30f6.jpg',
 'raptor__gbif_raptor_00254_jpg.rf.RUznLns9phCl4pmygjB7.jpg',
 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg',
 'raptor__raptor_043_jpg.rf.Zvj9EKGgySzsjUZnOEDi.jpg',
 'raptor__raptor_050_jpg.rf.duuf3tqrVk3ZzBLkisY3.jpg']

In [28]:
fn_images=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["pipeline"]=="CLOSE":

        fn_images.append(row["image"])

print("False Negative:",len(fn_images))

fn_images

False Negative: 9


['1054.jpeg',
 '1133.jpeg',
 '84.jpeg',
 'neg_poultry__poultry_082_jpg.rf.ampHW9A6XPI6kbuvpzqd.jpg',
 'neg_poultry__poultry_115_jpg.rf.n5StsXV8mhnHSm7Os8df.jpg',
 'neg_poultry__poultry_133_jpg.rf.Y9pO5YIarKaSPg8D9A7C.jpg',
 'OIP-HWyLJoTGMEY-Vr8B_l3bDQHaJ4.jpeg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg',
 'OIP-XHrr9AeiI8Egrt85dSx7mwHaFj.jpeg']

In [29]:
fp_yolo=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["yolo"]=="OPEN":

        fp_yolo.append(row["image"])

len(fp_yolo)

fp_yolo

['Image-84-bd2f1b.jpg', 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg']

In [30]:
fn_yolo=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["yolo"]=="CLOSE":

        fn_yolo.append(row["image"])

len(fn_yolo)

fn_yolo

['neg_poultry__poultry_222_jpg.rf.g0AgDML3T5PV9uIHp0Xd.jpg',
 'neg_poultry__poultry_244_jpg.rf.X1DkC0GjdbMdNEwCaauz.jpg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg']

In [31]:

OUTPUT = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset_experiments")



In [32]:
image_path_dark = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\dark\\images").glob("*"))
image_path_night = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\night\\images").glob("*"))
image_path_occlusion = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\occlusion\\images").glob("*"))
image_path_motion_blur = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\motion_blur\\images").glob("*"))

results_dark = []
results_night = []
results_occlusion = []
results_motion_blur = []


In [33]:
for image_path in image_path_dark:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_dark.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [34]:
for image_path in image_path_night:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_night.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [35]:
for image_path in image_path_occlusion:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_occlusion.append({
        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [36]:
for image_path in image_path_motion_blur:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_motion_blur.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [37]:
df_dark = pd.DataFrame(results_dark)
df_night = pd.DataFrame(results_night)
df_occlusion = pd.DataFrame(results_occlusion)
df_motion_blur = pd.DataFrame(results_motion_blur)

df_dark.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,24.6544,27.2610,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,26.7092,25.6969,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,20.2573,35.3746,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,18.2210,31.4308,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,21.3119,53.1785,"[0, 0, 0]","[0, 0, 0, 0, 0]","[0, 1, 0, 0, 1]"
5,1085.jpeg,OPEN,OPEN,CLOSE,22.3919,29.1249,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,18.9391,27.1113,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,21.4540,28.0127,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,22.4153,29.7936,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,CLOSE,19.3892,44.3184,"[0, 0, 0]","[0, 0, 0]","[0, 0, 1]"


In [38]:
df_night.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,29.7893,34.1001,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,17.6498,25.6824,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,22.7792,30.9689,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,19.7641,31.9447,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,22.4473,49.7604,"[0, 0, 0]","[0, 0, 0, 0]","[0, 1, 0, 0]"
5,1085.jpeg,OPEN,OPEN,CLOSE,20.7213,26.4895,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,24.0932,31.4611,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,20.1225,26.1158,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,18.8968,35.7065,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,CLOSE,14.1038,43.5372,"[0, 0, 0]","[0, 0, 0]","[0, 0, 1]"


In [39]:
df_occlusion.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,CLOSE,CLOSE,17.6226,17.2653,"[0, 0]",[],[]
1,1035.jpeg,OPEN,OPEN,OPEN,29.2141,34.0215,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,24.3882,46.1389,[0],"[0, 0, 0]","[0, 0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,20.8949,31.0946,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,23.1090,63.7073,"[0, 0, 0]","[0, 0, 0, 0, 0]","[0, 0, 1, 0, 1]"
5,1085.jpeg,OPEN,OPEN,CLOSE,27.1019,35.8384,[0],"[0, 0]","[1, 0]"
6,109.jpeg,OPEN,OPEN,OPEN,27.8867,30.1698,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,27.7861,31.7875,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,26.8946,30.7065,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,CLOSE,21.6236,52.1160,"[0, 0, 0]","[0, 0, 0, 0]","[0, 0, 1, 1]"


In [40]:
df_motion_blur.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,19.7598,29.1838,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,20.0034,30.8030,[0],[0],[0]
2,1048.jpeg,OPEN,CLOSE,CLOSE,18.4869,23.9058,[0],[],[]
3,1053.jpeg,OPEN,OPEN,OPEN,25.2160,27.4927,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,CLOSE,CLOSE,23.9408,33.2438,"[0, 0, 0]",[1],[1]
5,1085.jpeg,OPEN,OPEN,CLOSE,20.2518,32.0775,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,27.5694,25.4564,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,25.7456,31.9156,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,24.2093,33.4157,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,26.8524,29.5128,"[0, 0, 0]",[0],[0]


In [41]:
yolo_results1 = evaluate_system(
    df_dark,
    "yolo",
    "yolo_time_ms"
)
pipeline_results1 = evaluate_system(
    df_dark,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9886
Precision: 0.9897
Recall   : 0.9897
F1-score : 0.9897

TP : 482
FP : 5
TN : 386
FN : 5

Średni czas: 21.32 ms
pipeline
Accuracy : 0.9613
Precision: 0.9892
Recall   : 0.9405
F1-score : 0.9642

TP : 458
FP : 5
TN : 386
FN : 29

Średni czas: 33.24 ms


In [42]:
yolo_results2 = evaluate_system(
    df_night,
    "yolo",
    "yolo_time_ms"
)
pipeline_results2 = evaluate_system(
    df_night,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9727
Precision: 0.9874
Recall   : 0.9630
F1-score : 0.9751

TP : 469
FP : 6
TN : 385
FN : 18

Średni czas: 23.73 ms
pipeline
Accuracy : 0.9282
Precision: 0.9930
Recall   : 0.8768
F1-score : 0.9313

TP : 427
FP : 3
TN : 388
FN : 60

Średni czas: 35.73 ms


In [43]:
yolo_results3 = evaluate_system(
    df_occlusion,
    "yolo",
    "yolo_time_ms"
)
pipeline_results3 = evaluate_system(
    df_occlusion,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9795
Precision: 0.9916
Recall   : 0.9713
F1-score : 0.9813

TP : 473
FP : 4
TN : 387
FN : 14

Średni czas: 24.27 ms
pipeline
Accuracy : 0.9510
Precision: 0.9703
Recall   : 0.9405
F1-score : 0.9552

TP : 458
FP : 14
TN : 377
FN : 29

Średni czas: 38.15 ms


In [44]:
yolo_results4 = evaluate_system(
    df_motion_blur,
    "yolo",
    "yolo_time_ms"
)
pipeline_results4 = evaluate_system(
    df_motion_blur,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.8656
Precision: 0.9301
Recall   : 0.8193
F1-score : 0.8712

TP : 399
FP : 30
TN : 361
FN : 88

Średni czas: 20.76 ms
pipeline
Accuracy : 0.8838
Precision: 0.9661
Recall   : 0.8193
F1-score : 0.8867

TP : 399
FP : 14
TN : 377
FN : 88

Średni czas: 29.04 ms


In [45]:
comparison = pd.DataFrame(
    [
        yolo_results1,
        pipeline_results1
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.988610,0.989733,0.989733,0.989733,482,5,386,5,21.315957
YOLO + CNN,0.961276,0.989201,0.940452,0.964211,458,5,386,29,33.235342


In [46]:
comparison = pd.DataFrame(
    [
        yolo_results2,
        pipeline_results2

    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.972665,0.987368,0.963039,0.975052,469,6,385,18,23.730204
YOLO + CNN,0.928246,0.993023,0.876797,0.931298,427,3,388,60,35.727755


In [47]:
comparison = pd.DataFrame(
    [
        yolo_results3,
        pipeline_results3
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.979499,0.991614,0.971253,0.981328,473,4,387,14,24.273771
YOLO + CNN,0.951025,0.970339,0.940452,0.955162,458,14,377,29,38.152266


In [48]:
comparison = pd.DataFrame(
    [
        yolo_results4,
        pipeline_results4
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.865604,0.930070,0.819302,0.871179,399,30,361,88,20.763900
YOLO + CNN,0.883827,0.966102,0.819302,0.886667,399,14,377,88,29.035841


In [49]:
dangerous_yolo1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN")
]

In [50]:
dangerous_yolo2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN")
]

In [51]:
dangerous_yolo3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN")
]

In [52]:
dangerous_yolo4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN")
]

In [53]:
dangerous_pipeline1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["pipeline"]=="OPEN")
]

In [54]:
dangerous_pipeline2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["pipeline"]=="OPEN")
]

In [55]:
dangerous_pipeline3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["pipeline"]=="OPEN")
]

In [56]:
dangerous_pipeline4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]

In [57]:
dangerous_both = df[
    (df["ground_truth"]=="CLOSE") & (df["yolo"]=="OPEN") & (df["pipeline"]=="OPEN")
]
dangerous_both1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") & (df_dark["yolo"]=="OPEN") & (df_dark["pipeline"]=="OPEN")
]
dangerous_both2 = df_night[
    (df_night["ground_truth"]=="CLOSE") & (df_night["yolo"]=="OPEN") & (df_night["pipeline"]=="OPEN")
]
dangerous_both3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") & (df_occlusion["yolo"]=="OPEN") & (df_occlusion["pipeline"]=="OPEN")
]
dangerous_both4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") & (df_motion_blur["yolo"]=="OPEN") & (df_motion_blur["pipeline"]=="OPEN")
]


In [58]:
comparison = pd.DataFrame(

    [   

        yolo_results,

        pipeline_results,

        yolo_results1,

        pipeline_results1,

        yolo_results2,

        pipeline_results2,

        yolo_results3,

        pipeline_results3,

        yolo_results4,

        pipeline_results4

    ],

    index=[

        "YOLO normal",
        "YOLO+CNN normal",
        "YOLO dark",
        "YOLO+CNN dark",
        "YOLO night",
        "YOLO+CNN night",
        "YOLO occlusion",
        "YOLO+CNN occlusion",
        "YOLO motion",
        "YOLO+CNN motion"

    ]

)

comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO normal,0.994305,0.995885,0.993840,0.994861,484,2,389,3,20.145158
YOLO+CNN normal,0.981777,0.985567,0.981520,0.983539,478,7,384,9,30.774838
YOLO dark,0.988610,0.989733,0.989733,0.989733,482,5,386,5,21.315957
YOLO+CNN dark,0.961276,0.989201,0.940452,0.964211,458,5,386,29,33.235342
YOLO night,0.972665,0.987368,0.963039,0.975052,469,6,385,18,23.730204
YOLO+CNN night,0.928246,0.993023,0.876797,0.931298,427,3,388,60,35.727755
YOLO occlusion,0.979499,0.991614,0.971253,0.981328,473,4,387,14,24.273771
YOLO+CNN occlusion,0.951025,0.970339,0.940452,0.955162,458,14,377,29,38.152266
YOLO motion,0.865604,0.930070,0.819302,0.871179,399,30,361,88,20.763900
YOLO+CNN motion,0.883827,0.966102,0.819302,0.886667,399,14,377,88,29.035841


In [59]:
print()
print("Krytyczne błędy, wpuszczenie drapieżnika")
print("----------------")
print("YOLO dark:",len(dangerous_yolo1),"     YOLO night:",len(dangerous_yolo2),"     YOLO occlusion:",len(dangerous_yolo3),"    YOLO motion:",len(dangerous_yolo4))
print("YOLO+CNN dark:",len(dangerous_pipeline1),"YOLO+CNN night:",len(dangerous_pipeline2),"YOLO+CNN occlusion:",len(dangerous_pipeline3),"YOLO+CNN motion:",len(dangerous_pipeline4))
print("BOTH dark:",len(dangerous_both1),"     BOTH night:",len(dangerous_both2),"     BOTH occlusion:",len(dangerous_both3),"    BOTH motion:",len(dangerous_both4))


Krytyczne błędy, wpuszczenie drapieżnika
----------------
YOLO dark: 5      YOLO night: 6      YOLO occlusion: 4     YOLO motion: 30
YOLO+CNN dark: 5 YOLO+CNN night: 3 YOLO+CNN occlusion: 14 YOLO+CNN motion: 14
BOTH dark: 2      BOTH night: 2      BOTH occlusion: 3     BOTH motion: 5


In [60]:
locking_chicken_yolo = df[
    (df["ground_truth"]=="OPEN") & ((df["yolo"]=="CLOSE"))
]
locking_chicken_yolo1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["yolo"]=="CLOSE"))
]
locking_chicken_yolo2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["yolo"]=="CLOSE"))
]
locking_chicken_yolo3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["yolo"]=="CLOSE"))
]
locking_chicken_yolo4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["yolo"]=="CLOSE"))
]
locking_chicken_pipeline = df[
    (df["ground_truth"]=="OPEN") & ((df["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["pipeline"]=="CLOSE"))
]

locking_chicken_both = df[
    (df["ground_truth"]=="OPEN") & ((df["yolo"]=="CLOSE") | (df["pipeline"]=="CLOSE"))
]
locking_chicken_both1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["yolo"]=="CLOSE") | (df_dark["pipeline"]=="CLOSE"))
]
locking_chicken_both2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["yolo"]=="CLOSE") | (df_night["pipeline"]=="CLOSE"))
]
locking_chicken_both3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["yolo"]=="CLOSE") | (df_occlusion["pipeline"]=="CLOSE"))
]
locking_chicken_both4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["yolo"]=="CLOSE") | (df_motion_blur["pipeline"]=="CLOSE"))
]

In [61]:
print()
print("Niekrytyczne błędy, niewpuszczenie kur")
print("----------------")
print("YOLO dark:",len(locking_chicken_both1),"     YOLO night:",len(locking_chicken_both2),"     YOLO occlusion:",len(locking_chicken_both3),"    YOLO motion:",len(locking_chicken_both4))



Niekrytyczne błędy, niewpuszczenie kur
----------------
YOLO dark: 30      YOLO night: 60      YOLO occlusion: 30     YOLO motion: 101


In [62]:
data = [[len(dangerous_yolo), len(dangerous_yolo1), len(dangerous_yolo2), len(dangerous_yolo3), len(dangerous_yolo4)],
        [len(dangerous_pipeline), len(dangerous_pipeline1), len(dangerous_pipeline2), len(dangerous_pipeline3), len(dangerous_pipeline4)],
        [len(dangerous_both), len(dangerous_both1), len(dangerous_both2), len(dangerous_both3), len(dangerous_both4)]]
columns = ["normal", "dark", "night", "occlusion", "motion"]
index = ["yolo", "yolo+cnn", "both"]
table = pd.DataFrame(data, columns=columns, index=index)
tolatextable = table.to_latex(index=True, float_format="{:.2f}".format,caption = "placeholder", label = "placeholder", position = "!h")
print("\\setlength{\\tabcolsep}{6pt}")
print("\\multicolumn{5}{c}{\\textbf{Krytyczne błędy, wpuszczenie przeciwnika}} ")
print(tolatextable)

\setlength{\tabcolsep}{6pt}
\multicolumn{5}{c}{\textbf{Krytyczne błędy, wpuszczenie przeciwnika}} 
\begin{table}[!h]
\caption{placeholder}
\label{placeholder}
\begin{tabular}{lrrrrr}
\toprule
 & normal & dark & night & occlusion & motion \\
\midrule
yolo & 2 & 5 & 6 & 4 & 30 \\
yolo+cnn & 7 & 5 & 3 & 14 & 14 \\
both & 1 & 2 & 2 & 3 & 5 \\
\bottomrule
\end{tabular}
\end{table}



In [63]:
data_locking = [[len(locking_chicken_yolo), len(locking_chicken_yolo1), len(locking_chicken_yolo2), len(locking_chicken_yolo3), len(locking_chicken_yolo4)],
    [len(locking_chicken_pipeline), len(locking_chicken_pipeline1), len(locking_chicken_pipeline2), len(locking_chicken_pipeline3), len(locking_chicken_pipeline4)],
    [len(locking_chicken_both), len(locking_chicken_both1), len(locking_chicken_both2), len(locking_chicken_both3), len(locking_chicken_both4)]]

columns_lock = ["normal", "dark", "night", "occlusion", "motion"]
index_lock = ["yolo", "yolo+cnn", "both"]

table2 = pd.DataFrame(data_locking, columns=columns, index=index)

tolatextable2 = table2.to_latex(index=True, float_format="{:.2f}".format,caption = "placeholder", label = "placeholder", position = "!h")
print("\\setlength{\\tabcolsep}{6pt}")
print("\\multicolumn{5}{c}{\\textbf{Niekrytycznie błędy, niewpuszczenie kur}} ")
print(tolatextable2)

\setlength{\tabcolsep}{6pt}
\multicolumn{5}{c}{\textbf{Niekrytycznie błędy, niewpuszczenie kur}} 
\begin{table}[!h]
\caption{placeholder}
\label{placeholder}
\begin{tabular}{lrrrrr}
\toprule
 & normal & dark & night & occlusion & motion \\
\midrule
yolo & 3 & 5 & 18 & 14 & 88 \\
yolo+cnn & 9 & 29 & 60 & 29 & 88 \\
both & 11 & 30 & 60 & 30 & 101 \\
\bottomrule
\end{tabular}
\end{table}



In [64]:
latex_table = comparison.to_latex(index=True, float_format="{:.2f}".format,caption = "placeholder", label = "placeholder", position = "!h")
print("\\setlength{\\tabcolsep}{6pt}")
print(latex_table)

\setlength{\tabcolsep}{6pt}
\begin{table}[!h]
\caption{placeholder}
\label{placeholder}
\begin{tabular}{lrrrrrrrrr}
\toprule
 & Accuracy & Precision & Recall & F1 & TP & FP & TN & FN & Time \\
\midrule
YOLO normal & 0.99 & 1.00 & 0.99 & 0.99 & 484 & 2 & 389 & 3 & 20.15 \\
YOLO+CNN normal & 0.98 & 0.99 & 0.98 & 0.98 & 478 & 7 & 384 & 9 & 30.77 \\
YOLO dark & 0.99 & 0.99 & 0.99 & 0.99 & 482 & 5 & 386 & 5 & 21.32 \\
YOLO+CNN dark & 0.96 & 0.99 & 0.94 & 0.96 & 458 & 5 & 386 & 29 & 33.24 \\
YOLO night & 0.97 & 0.99 & 0.96 & 0.98 & 469 & 6 & 385 & 18 & 23.73 \\
YOLO+CNN night & 0.93 & 0.99 & 0.88 & 0.93 & 427 & 3 & 388 & 60 & 35.73 \\
YOLO occlusion & 0.98 & 0.99 & 0.97 & 0.98 & 473 & 4 & 387 & 14 & 24.27 \\
YOLO+CNN occlusion & 0.95 & 0.97 & 0.94 & 0.96 & 458 & 14 & 377 & 29 & 38.15 \\
YOLO motion & 0.87 & 0.93 & 0.82 & 0.87 & 399 & 30 & 361 & 88 & 20.76 \\
YOLO+CNN motion & 0.88 & 0.97 & 0.82 & 0.89 & 399 & 14 & 377 & 88 & 29.04 \\
\bottomrule
\end{tabular}
\end{table}



In [65]:
improved = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN") &
    (df_dark["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,20.4012,26.7411,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,18.8045,29.3890,[1],"[0, 0]","[1, 1]"
833,raptor__gbif_raptor_00646_jpg.rf.IC8W8N97wC0S3...,CLOSE,OPEN,CLOSE,25.2045,27.7467,[1],[0],[1]


In [66]:
improved = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN") &
    (df_night["pipeline"]=="CLOSE")

]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
285,Image-51-eb7770.jpg,CLOSE,OPEN,CLOSE,17.1998,24.4173,[1],[0],[1]
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,30.5977,30.4050,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,25.3572,36.1760,[1],"[0, 0]","[1, 1]"
833,raptor__gbif_raptor_00646_jpg.rf.IC8W8N97wC0S3...,CLOSE,OPEN,CLOSE,22.5896,23.4019,[1],[0],[1]


In [67]:
improved = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN") &
    (df_occlusion["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,24.9727,31.6535,[1],[0],[1]


In [68]:
improved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN") &
    (df_motion_blur["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
91,coyote__lila_AMMonitor_Camera_Traps_NEK-VB1_00...,CLOSE,OPEN,CLOSE,21.2087,23.6363,[1],[0],[1]
141,coyote__lila_WSU_Lynx_IMG_0921_jpg.rf.LXZgTLOf...,CLOSE,OPEN,CLOSE,20.4808,34.8949,[1],[0],[1]
166,fox__gbif_fox_0058_jpg.rf.276TrYjTZ8l8fHW2cbDV...,CLOSE,OPEN,CLOSE,26.0617,24.7067,[1],[0],[1]
186,fox__gbif_fox_0454_jpg.rf.ET1jFfTjXFmpusKl9rJT...,CLOSE,OPEN,CLOSE,19.2031,36.2845,[1],[0],[1]
193,fox__gbif_fox_0540_jpg.rf.SviRhT8VBkdRDGFU1QO6...,CLOSE,OPEN,CLOSE,22.9392,28.4325,[1],[0],[1]
196,fox__gbif_fox_0615_jpg.rf.xGBhUyQrn1rpiysNE7WH...,CLOSE,OPEN,CLOSE,21.8138,34.0215,[1],[0],[1]
198,fox__gbif_fox_0722_jpg.rf.4EaY143B8mKPFC0058JJ...,CLOSE,OPEN,CLOSE,27.3758,36.8534,[1],[0],[1]
226,Image-111-ed55ed.jpg,CLOSE,OPEN,CLOSE,24.0454,30.6393,[1],[0],[1]
230,Image-115-0540fa.png,CLOSE,OPEN,CLOSE,31.7716,43.9253,[1],[0],[1]
250,Image-31-1d307b.jpg,CLOSE,OPEN,CLOSE,25.2736,28.5641,[1],[0],[1]


In [69]:
deproved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]
deproved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
162,fox__gbif_fox_0007_jpg.rf.2EDZp1rkJPFaJAxi0jyV...,CLOSE,CLOSE,OPEN,27.4690,31.0754,[1],[1],[0]
171,fox__gbif_fox_0117_jpg.rf.qu0oF3WRaQtW6UDSh1cN...,CLOSE,CLOSE,OPEN,29.1982,35.7285,[1],[1],[0]
213,Image-10-95384c.jpg,CLOSE,CLOSE,OPEN,30.1498,44.2054,[1],"[1, 1]","[0, 0]"
245,Image-28-88c850.jpg,CLOSE,CLOSE,OPEN,33.0260,51.7836,[1],"[1, 1]","[0, 0]"
327,Image-82-7d6856.jpg,CLOSE,CLOSE,OPEN,16.6006,27.0131,[1],[1],[0]
355,neg_empty__background_empty_043_jpg.rf.XPpXqOG...,CLOSE,CLOSE,OPEN,24.6783,31.3473,[],[1],[0]
375,neg_empty__background_empty_190_jpg.rf.jE7KIPf...,CLOSE,CLOSE,OPEN,24.8766,37.7912,[],[1],[0]
378,neg_empty__background_empty_215_jpg.rf.Dcj5ZaN...,CLOSE,CLOSE,OPEN,17.4553,29.6203,[],[1],[0]
874,raptor__raptor_027_jpg.rf.PzIAqRQVSQEonrCHFVSD...,CLOSE,CLOSE,OPEN,17.1791,41.6135,"[1, 1]",[1],[0]


In [70]:
empty_images0 = 0
false_detections0 = 0
for row in results_dark:

    if len(row["objects_gt"]) == 0:

        empty_images0 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections0 += 1

print("Puste obrazy:",empty_images0)
print("Fałszywe detekcje:",false_detections0)
if empty_images0>0:

    print(
        "Odsetek:",
        false_detections0/empty_images0
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [71]:
empty_images1 = 0
false_detections1 = 0

for row in results_night:

    if len(row["objects_gt"]) == 0:

        empty_images1 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections1 += 1

print("Puste obrazy:",empty_images1)
print("Fałszywe detekcje:",false_detections1)

if empty_images1>0:

    print(
        "Odsetek:",
        false_detections1/empty_images1
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [72]:
empty_images2 = 0
false_detections2 = 0

for row in results_occlusion:

    if len(row["objects_gt"]) == 0:

        empty_images2 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections2 += 1

print("Puste obrazy:",empty_images2)
print("Fałszywe detekcje:",false_detections2)

if empty_images2>0:

    print(
        "Odsetek:",
        false_detections2/empty_images2
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [73]:
empty_images3 = 0
false_detections3 = 0

for row in results_motion_blur:

    if len(row["objects_gt"]) == 0:

        empty_images3 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections3 += 1

print("Puste obrazy:",empty_images3)
print("Fałszywe detekcje:",false_detections3)

if empty_images3>0:

    print(
        "Odsetek:",
        false_detections3/empty_images3
    )

Puste obrazy: 37
Fałszywe detekcje: 7
Odsetek: 0.1891891891891892


In [74]:

#ZMIEŃ CONFIDENC POTEM NA 0,5 I PORÓWNAJ!!!!!!